# Mock 5A · Solar Filament Segmentation · Phân đoạn sợi plasma (filament) trên ảnh Mặt Trời (Computer Vision · Thị giác máy tính)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Each image is a full-disk photo of the Sun in the H-alpha line, taken by the GONG telescope network. **Filaments** are clouds of cooler plasma held above the surface by magnetic fields; they look like thin dark threads on the disc. Predict, for every pixel, whether it belongs to a filament (a binary mask). | Mỗi ảnh là ảnh toàn đĩa Mặt Trời ở vạch H-alpha do mạng kính thiên văn GONG chụp. **Filament** là những đám plasma nguội hơn được từ trường giữ lơ lửng trên bề mặt; trên đĩa chúng trông như những sợi tối, mảnh. Hãy dự đoán với từng điểm ảnh xem nó có thuộc filament hay không (mặt nạ nhị phân). |
| **Data** | `train/` 472 images **with masks** (observed 2011-2015) · `public_test/` and `private_test/` images without masks, observed **2016-2022**: later years, a different phase of the solar cycle. All images and masks are **1024×1024**, greyscale. `train.csv` / `<split>.csv` give the observing station (`site`). | `train/` 472 ảnh **có mặt nạ** (chụp 2011-2015) · `public_test/` và `private_test/` là ảnh không có mặt nạ, chụp **2016-2022**: những năm sau, ở pha khác của chu kỳ Mặt Trời. Mọi ảnh và mặt nạ đều **1024×1024**, ảnh xám. `train.csv` / `<split>.csv` cho biết trạm quan sát (`site`). |
| **Metric** | Mean per-image **Dice** = 2·\|P∩T\| / (\|P\|+\|T\|). An image with **no filament** scores 1 if you predict an empty mask and 0 otherwise. **Higher is better.** Ranking uses the **private** score. | **Dice** trung bình theo ảnh = 2·\|P∩T\| / (\|P\|+\|T\|). Ảnh **không có filament** được 1 điểm nếu bạn dự đoán mặt nạ rỗng, ngược lại 0 điểm. **Càng cao càng tốt.** Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,rle`: one row per test image, the mask at **1024×1024** in run-length encoding (`rle_encode` below: pixels numbered from 1, top-to-bottom then left-to-right, `start length` pairs). Empty string = no filament. | `public_submission.csv`, `private_submission.csv` gồm cột `id,rle`: mỗi ảnh test một dòng, mặt nạ ở kích thước **1024×1024** mã hoá run-length (`rle_encode` bên dưới: điểm ảnh đánh số từ 1, từ trên xuống rồi từ trái sang phải, các cặp `start length`). Chuỗi rỗng = không có filament. |
| **Rules** | PyTorch. ImageNet-pretrained backbones from `torchvision` / `timm` are allowed. **No external solar data**, and **do not use the original MAGFiLO annotations or the Kaggle filament-segmentation data** (the test images are real MAGFiLO observations: honour code). No hand-labelling of test images. | PyTorch. Được dùng backbone huấn luyện sẵn trên ImageNet từ `torchvision` / `timm`. **Không dùng dữ liệu Mặt Trời bên ngoài**, và **không dùng nhãn MAGFiLO gốc hay dữ liệu filament của Kaggle** (ảnh test là quan sát MAGFiLO thật: hãy giữ danh dự). Không tự gán nhãn ảnh test. |

$$\text{Score}=\frac{1}{N}\sum_{i=1}^{N}\text{Dice}(P_i,T_i),\qquad \text{Dice}(\varnothing,\varnothing)=1$$

**Files · Các file:**
- `train/images/<id>.jpg`, `train/masks/<id>.png` (0 = background · nền, 255 = filament)
- `train/train.csv` — `id, site, year, n_annotators, filament_pixels` (`n_annotators`: how many people annotated the image; where several did, a pixel is a filament when at least half of them drew it · số người gán nhãn ảnh; khi có nhiều người, một điểm ảnh là filament nếu ít nhất một nửa số người vẽ nó)
- `public_test/images/`, `public_test/public_test.csv`, `private_test/images/`, `private_test/private_test.csv` — `id, site`

🇬🇧 Not every dark feature is a filament: sunspots and dark patches near active regions are not. Filaments are thin, so a few pixels of misalignment cost a lot of Dice, and the human annotators themselves disagree on faint ones.
🇻🇳 Không phải vệt tối nào cũng là filament: vết đen Mặt Trời và các mảng tối quanh vùng hoạt động thì không. Filament rất mảnh nên lệch vài điểm ảnh đã mất nhiều Dice, và chính người gán nhãn cũng không thống nhất với những sợi mờ.

Data · Dữ liệu: images GONG H-alpha, NSO/AURA/NSF (CC BY 4.0); masks derived from [MAGFiLO v1.0](https://doi.org/10.7910/DVN/J6JNVK), Ahmadzadeh et al. 2024 (CC BY-NC 4.0). See `dataset/README.md`.
Inspired by · Lấy cảm hứng từ: [IEEE BigData 2026 Solar Filament Segmentation (Kaggle)](https://www.kaggle.com/competitions/filament-segmentation-2026).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt
import torch.nn.functional as F
from PIL import Image

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "5A"
FOLDER  = "5A_solar_filament"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-5"

# Already have the folder? Point this at the parent of 5A_solar_filament/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 5A_solar_filament/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/5A_solar_filament/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
SIZE = 1024  # images, masks and submissions / ảnh, mặt nạ và file nộp
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
IMG_SIZE = 384    # training/prediction resolution (images are 1024) / độ phân giải khi huấn luyện, dự đoán
BASE_CH = 16      # channels of the first U-Net level / số kênh tầng đầu của U-Net
BATCH_SIZE = 8
EPOCHS = 12
LR = 1e-3
THRESHOLD = 0.5   # probability -> mask / ngưỡng xác suất -> mặt nạ
POS_WEIGHT = 10.0  # filaments are < 1% of the pixels: weight them up in the loss / filament < 1% điểm ảnh nên tăng trọng số
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Visualisation · Đọc và hiển thị dữ liệu

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
public_df = pd.read_csv(PUBLIC_CSV)
private_df = pd.read_csv(PRIVATE_CSV)
print(f"Train: {len(train_df)} | Public test: {len(public_df)} | Private test: {len(private_df)}")
print("Images per station / Số ảnh theo trạm:", train_df["site"].value_counts().to_dict())


def rle_encode(mask):
    """EN: binary mask (H, W) -> Kaggle RLE string: pixels numbered 1..H*W top-to-bottom, then left-to-right.
    VI: mặt nạ nhị phân (H, W) -> chuỗi RLE kiểu Kaggle: đánh số điểm ảnh 1..H*W từ trên xuống, rồi trái sang phải."""
    flat = np.asarray(mask, dtype=bool).flatten(order="F").astype(np.int8)
    padded = np.concatenate([[0], flat, [0]])
    runs = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))


def rle_decode(rle, shape=(SIZE, SIZE)):
    """EN: inverse of rle_encode. / VI: ngược lại của rle_encode."""
    flat = np.zeros(shape[0] * shape[1], dtype=bool)
    if isinstance(rle, str) and rle.strip():
        nums = np.asarray(rle.split(), dtype=np.int64)
        for s, n in zip(nums[0::2], nums[1::2]):
            flat[s - 1:s - 1 + n] = True
    return flat.reshape(shape, order="F")


def load_images(split, ids, size=IMG_SIZE):
    """EN: read greyscale images and shrink them to size x size -> uint8 array (N, size, size).
    VI: đọc ảnh xám và thu nhỏ về size x size -> mảng uint8 (N, size, size)."""
    return np.stack([np.asarray(Image.open(f"{DATA_DIR}/{split}/images/{i}.jpg").convert("L")
                                .resize((size, size), Image.Resampling.BILINEAR)) for i in ids])


def load_masks(ids, size=IMG_SIZE):
    """EN: masks shrunk the same way; a small pixel counts as filament if any part of it was.
    VI: mặt nạ thu nhỏ tương tự; một điểm ảnh nhỏ là filament nếu có phần nào thuộc filament."""
    return np.stack([(np.asarray(Image.open(f"{DATA_DIR}/train/masks/{i}.png")
                                 .resize((size, size), Image.Resampling.BOX)) > 0).astype(np.float32) for i in ids])


X_all = load_images("train", train_df["id"])
Y_all = load_masks(train_df["id"])
X_public = load_images("public_test", public_df["id"])
X_private = load_images("private_test", private_df["id"])
print(f"Images in memory / Ảnh trong bộ nhớ: {X_all.shape}, filament pixels / điểm ảnh filament: {Y_all.mean():.4%}")

# EN: random 10% validation split / VI: tách ngẫu nhiên 10% làm tập validation
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X_all))
n_val = len(X_all) // 10
val_idx, tr_idx = perm[:n_val], perm[n_val:]


def to_tensor(x):
    return torch.from_numpy(x).float().div(255.0).unsqueeze(1)  # (N, 1, H, W) in [0, 1]


train_loader = DataLoader(TensorDataset(to_tensor(X_all[tr_idx]), torch.from_numpy(Y_all[tr_idx]).unsqueeze(1)),
                          batch_size=BATCH_SIZE, shuffle=True)
print(f"Train: {len(tr_idx)} | Validation: {len(val_idx)}")

# EN: a few training images with their filaments in red / VI: vài ảnh train, filament tô đỏ
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, i in zip(axes, rng.choice(len(X_all), 3, replace=False)):
    rgb = np.repeat(X_all[i][..., None], 3, axis=2).copy()
    rgb[Y_all[i] > 0] = [255, 0, 0]
    ax.imshow(rgb)
    ax.set_title(f"{train_df['id'][i]} · site {train_df['site'][i]} · {train_df['year'][i]}")
    ax.axis("off")
plt.show()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
def block(c_in, c_out):
    """EN: two 3x3 convolutions with batch norm. / VI: hai lớp tích chập 3x3 kèm batch norm."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """EN: small U-Net trained from scratch: 3 down-sampling steps, skip connections, 1 output logit per pixel.
    VI: U-Net nhỏ huấn luyện từ đầu: 3 lần giảm kích thước, kết nối tắt, 1 logit cho mỗi điểm ảnh."""

    def __init__(self, c=BASE_CH):
        super().__init__()
        self.enc1, self.enc2, self.enc3 = block(1, c), block(c, 2 * c), block(2 * c, 4 * c)
        self.mid = block(4 * c, 8 * c)
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * c, 4 * c, 2, stride=2), block(8 * c, 4 * c)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2), block(4 * c, 2 * c)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2), block(2 * c, c)
        self.out = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        m = self.mid(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(m), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


model = UNet().to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict_masks(model, x_uint8, out_size=SIZE, threshold=THRESHOLD):
    """EN: images (N, IMG_SIZE, IMG_SIZE) -> boolean masks (N, out_size, out_size): the probabilities are
    resized back to the full resolution, then thresholded.
    VI: ảnh (N, IMG_SIZE, IMG_SIZE) -> mặt nạ (N, out_size, out_size): phóng xác suất về độ phân giải gốc rồi cắt ngưỡng."""
    model.eval()
    out = []
    for i in range(0, len(x_uint8), 8):
        xb = to_tensor(x_uint8[i:i + 8]).to(DEVICE)
        prob = torch.sigmoid(model(xb))
        prob = F.interpolate(prob, size=(out_size, out_size), mode="bilinear", align_corners=False)
        out.append((prob[:, 0] > threshold).cpu().numpy())
    return np.concatenate(out)


def mean_dice(pred_masks, true_masks):
    """EN: the competition metric (an empty truth scores 1 only with an empty prediction).
    VI: độ đo của cuộc thi (ảnh không có filament chỉ được 1 điểm khi dự đoán rỗng)."""
    scores = []
    for p, t in zip(pred_masks, true_masks):
        s = p.sum() + t.sum()
        scores.append(1.0 if s == 0 else 2.0 * np.logical_and(p, t).sum() / s)
    return float(np.mean(scores))


# EN: validation is scored on the full 1024x1024 masks, exactly like the leaderboard
# VI: validation được chấm trên mặt nạ 1024x1024 đầy đủ, giống hệt bảng xếp hạng
val_true = np.stack([np.asarray(Image.open(f"{DATA_DIR}/train/masks/{i}.png")) > 0
                     for i in train_df["id"].iloc[val_idx]])

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))  # logits / mô hình trả về logit
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_dice = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, t0 = 0.0, time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    val_dice = mean_dice(predict_masks(model, X_all[val_idx]), val_true)
    msg = (f"Epoch {epoch:02d} | Loss={total_loss / len(tr_idx):.4f} | Val mean Dice={val_dice:.4f} "
           f"| {time.time() - t0:.0f}s")
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation mean Dice: {best_dice:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, masks, csv_path):
    """Save <id, rle> CSV (masks at 1024x1024) and zip it. / Lưu CSV <id, rle> (mặt nạ 1024x1024) và nén zip."""
    pd.DataFrame({"id": ids, "rle": [rle_encode(m) for m in masks]}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
public_masks = predict_masks(model, X_public)
private_masks = predict_masks(model, X_private)
print(f"Empty predicted masks / Mặt nạ rỗng: public {np.mean([m.sum() == 0 for m in public_masks]):.0%}, "
      f"private {np.mean([m.sum() == 0 for m in private_masks]):.0%}")
export(public_df["id"], public_masks, "public_submission.csv")
export(private_df["id"], private_masks, "private_submission.csv")
pd.read_csv("public_submission.csv", keep_default_na=False).head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path, keep_default_na=False, dtype={"rle": str})
    assert list(sub.columns) == ["id", "rle"], f"{csv_path}: columns must be ['id', 'rle']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    for i, rle in zip(sub["id"], sub["rle"]):
        nums = np.asarray(rle.split(), dtype=np.int64)
        assert len(nums) % 2 == 0, f"{csv_path}: id {i}: rle needs 'start length' pairs / cần các cặp 'start length'"
        starts, lengths = nums[0::2], nums[1::2]
        assert (starts >= 1).all() and (lengths >= 1).all(), f"{csv_path}: id {i}: starts are 1-based, lengths >= 1"
        assert (starts + lengths - 1 <= SIZE * SIZE).all(), f"{csv_path}: id {i}: mask must be {SIZE}x{SIZE}"
        assert (starts[1:] > (starts + lengths - 1)[:-1]).all(), f"{csv_path}: id {i}: runs must be sorted, no overlap"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "5A"          # which problem this notebook is for
CONTEST = "mock-5"    # this notebook is for Mock Contest 5
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")